# Schwarzschild propagation and finite-time recovery

This notebook isolates how the complex Schwarzschild transmission channel affects the existing Hayden–Preskill diary calculation. It regenerates the 32 source states and compares three propagation conditions while holding the emitted packets, observer, receiver, and collection deadline fixed.

The source emission phases are calibrated **once**, using the baseline Schwarzschild calculation. They are then unchanged in every condition and refinement. Neither transmitted packets nor collected probabilities are renormalized.

| Condition | Interpretation |
| --- | --- |
| Schwarzschild | Full complex scalar transmission through the radial curvature potential. |
| Unit transmission | A potential-free transfer reference with the same source phases and observer lapse. |
| Magnitude only | The same spectral attenuation with transmission phase removed. This mathematical diagnostic need not define a causal propagation channel. |

The receiver starts empty at proper time zero and collects until $120M$ in every condition. The comparison concerns the declared finite observation window; it does not claim compactly supported emission or a standalone travel-time measurement.

In [1]:
from pathlib import Path
import os, datetime
import sys
import numpy as np
from IPython.display import display
sys.path.insert(0,str(Path('code').resolve()))
from schwarzschild_controls import run_controls
control_out=Path('outputs/runs')/os.environ.get('ODER_RUN_NAME',datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))/'controls'
tables, settings = run_controls(output=control_out)
print('Source phase calibration delay:',settings['source_calibration_delay_M'],'M')
print('Frozen source coordinate centers:',settings['frozen_source_coordinate_centers_M'])
display(tables['solver'])

baseline controls complete


refined controls complete


Source phase calibration delay: 5.041783756794742 M
Frozen source coordinate centers: [24.200084781067602, 36.84919542174112, 49.498306062414635, 62.147416703088155, 74.79652734376167]


,setting,frequency_nodes,time_step_M,spectral_points,maximum_flux_residual,seconds
0,baseline,321,0.10,2401,1.358402e-09,31.339657
1,refined,641,0.05,4801,4.327072e-11,59.967468


## Collection and recovery at the same deadline

The source tables are contracted separately for every seed. The information columns are $I(Q:X)/(2S(Q))$, with no early radiation or with all early radiation. The fidelity ceiling is the largest no-early ceiling among the 32 seeds; the floor is the smallest full-early floor. These are recovery bounds for the collected register.

Spectral transmission measures the full packet energy admitted by the transfer function. In-window incident and collected energies also depend on packet timing and the common initial condition. Collection includes the detector output and the remaining cavity occupation at the deadline.

In [2]:
display(tables['summary'])
display(tables['packets'])
base=tables['summary'].query("setting == 'baseline'").set_index('condition')
assert base.loc['schwarzschild','fidelity_ceiling'] < .3572
assert base.loc['schwarzschild','fidelity_floor'] > .8155
assert abs(base.loc['schwarzschild','spectral_transmission']-base.loc['magnitude_only','spectral_transmission']) < 1e-12
assert np.all(tables['states']['floor'] <= 1+2e-12)
assert np.all(tables['states']['ceiling'] >= .25-2e-12)

,setting,condition,spectral_transmission,input_band_mass,mean_terminal_incident,mean_terminal_collected,mean_no_early,mean_full_early,fidelity_ceiling,fidelity_floor,maximum_energy_balance_residual
0,baseline,schwarzschild,0.722212,1.0,0.715318,0.445132,0.018876,0.856854,0.357162,0.815598,3.750472e-15
1,baseline,unit_transmission,1.000000,1.0,0.985768,0.611494,0.050967,0.948735,0.427892,0.927765,4.213643e-15
2,baseline,magnitude_only,0.722212,1.0,0.708848,0.440029,0.018230,0.852872,0.355357,0.811060,3.210973e-15
3,refined,schwarzschild,0.722212,1.0,0.715318,0.445132,0.018876,0.856854,0.357162,0.815598,2.076464e-15
4,refined,unit_transmission,1.000000,1.0,0.985768,0.611494,0.050967,0.948735,0.427892,0.927765,4.014150e-15
5,refined,magnitude_only,0.722212,1.0,0.708848,0.440029,0.018230,0.852872,0.355357,0.811060,2.218711e-15


,setting,condition,packet,source_coordinate_center_M,input_peak_M,terminal_incident,terminal_collected,terminal_cavity
0,baseline,schwarzschild,1,24.200085,28.000003,0.694831,0.428148,9.927422e-08
1,baseline,schwarzschild,2,36.849195,40.000003,0.718247,0.447060,1.746861e-06
2,baseline,schwarzschild,3,49.498306,52.000003,0.721869,0.450370,2.688804e-05
3,baseline,schwarzschild,4,62.147417,64.000003,0.721960,0.450610,2.999328e-04
4,baseline,schwarzschild,5,74.796527,76.000003,0.719684,0.449470,2.180099e-03
5,baseline,unit_transmission,1,24.200085,22.958216,0.939309,0.573895,3.685015e-09
6,baseline,unit_transmission,2,36.849195,34.958216,0.990832,0.614844,4.547777e-08
7,baseline,unit_transmission,3,49.498306,46.958216,0.999226,0.622431,1.835953e-06
8,baseline,unit_transmission,4,62.147417,58.958216,0.999946,0.623202,5.133601e-05
9,baseline,unit_transmission,5,74.796527,70.958216,0.999526,0.623096,7.277938e-04


## Numerical refinement and the beginning of the observation window

The refinement simultaneously doubles the transmission grid, halves the time step, doubles the packet quadrature, tightens the radial ODE tolerances, and extends its boundaries. Source phases remain fixed. The table reports the resulting numerical changes; these are convergence diagnostics, not statistical confidence intervals.

Positive-frequency Gaussian packets have tails before the observation window. The final table reports the first packet's incident energy on $[-800M,0]$ and the remote contribution on $[-800M,-400M]$. These quantities are diagnostics only: all receiver integrations still start with an empty cavity at zero. Later packets have the same shapes shifted by the common emission schedule.

In [3]:
display(tables['refinement'])
display(tables['prewindow'])
assert tables['refinement']['maximum_terminal_probability_change'].max() < 8e-4
assert tables['refinement']['maximum_information_change'].max() < 1.5e-3
print('All fixed-source controls passed. Tables and arrays:',control_out)

,condition,maximum_terminal_probability_change,maximum_information_change,maximum_fidelity_bound_change
0,schwarzschild,5.176198e-07,1.710150e-07,1.048455e-07
1,unit_transmission,2.555119e-10,2.380133e-10,8.608275e-11
2,magnitude_only,3.163141e-09,1.981580e-09,1.290498e-09


,condition,packet,incident_energy_minus800_to0,incident_energy_minus800_to_minus400
0,schwarzschild,1,0.027382,4.235089e-12
1,unit_transmission,1,0.060691,1.471684e-08
2,magnitude_only,1,0.054410,4.153372e-12


All fixed-source controls passed. Tables and arrays: outputs\runs\companion_20261103_complete\controls


## Check the time-cut information bound

Whole-packet transmission and information carried through the finite observation window are different quantities. The following check contracts the same source states with the integrated in-window incident probabilities to obtain $e_Q(120M)$. For every seed and condition, observer information must not exceed this time-cut supply.

In [4]:
import json
import pandas as pd
from cavity import mask_weights
# control_out is the fresh output directory selected above.
source=np.load(control_out/'control_source_information.npy')
supply_rows=[]
for setting in ('baseline','refined'):
    for condition in ('schwarzschild','unit_transmission','magnitude_only'):
        data=np.load(control_out/f'control_{setting}_{condition}.npz')
        supply=np.einsum('d,sd->s',mask_weights(data['global_carriers'][-1]),source[:,:,31])
        observer_info=data['information'][:,31]
        assert np.max(observer_info-supply)<2e-9
        supply_rows.append(dict(setting=setting,condition=condition,
            mean_time_cut_supply=float(supply.mean()),
            maximum_statewise_nesting_difference=float(np.max(observer_info-supply))))
supply_table=pd.DataFrame(supply_rows)
display(supply_table)
supply_table.to_csv(control_out/'control_time_cut_supply.csv',index=False)
report=json.loads((control_out/'control_summary.json').read_text())
report['time_cut_supply']=supply_rows
(control_out/'control_summary.json').write_text(json.dumps(report,indent=2)+'\n')
print('Time-cut information nesting holds for every source state and control.')

,setting,condition,mean_time_cut_supply,maximum_statewise_nesting_difference
0,baseline,schwarzschild,0.976589,-0.118163
1,baseline,unit_transmission,0.999037,-0.048968
2,baseline,magnitude_only,0.975360,-0.120895
3,refined,schwarzschild,0.976589,-0.118163
4,refined,unit_transmission,0.999037,-0.048968
5,refined,magnitude_only,0.975360,-0.120895


Time-cut information nesting holds for every source state and control.


## Reading the comparison

The unit-transmission reference tests the combined effect of the Schwarzschild transfer function on the specified finite observer. The magnitude-only diagnostic separates a phase-sensitive contribution at the same spectral attenuation. Neither comparison changes possession of the early radiation, the diary encoding, or the detector settings.

The comparisons do not constitute a Hawking-production calculation. They quantify the role of propagation in the paper's prepared-diary recovery task. The finite-window results should be read together with their source phase convention, packet prehistory, and numerical refinement.

[Causal phase and delay](Causal_Phase_and_Delay.ipynb) evaluates the separate routed element downstream of the Schwarzschild scatterer. It compares a dispersive arm with a matched pure delay and accounts explicitly for upstream prehistory and stored energy.
